# Basic Workflow: Processing Raw Text with `inif`

This notebook demonstrates the basic `inif` workflow for processing raw text inputs with a base LLM tokenizer.

**Scenario:** A researcher has several text passages and wants to:
1. Tokenize them using a model's tokenizer
2. Find common token sequences across passages
3. Tag tokens of interest (e.g. numbers, named entities)
4. Store mock interpretability data (e.g. logit lens) on tagged tokens
5. Save and reload the enriched document

For a workflow starting from an **Inspect AI** evaluation log, see `inspect_workflow.ipynb`.

## Step 1: Convert raw text to inif format

The `from_texts` converter tokenizes each text string into a `Sample`, finds common token sequences across samples via set-intersection, and replaces them with compact sequence references.

We pass the HuggingFace model ID `"gpt2"` as a string — the converter loads the tokenizer via `AutoTokenizer.from_pretrained`. You can also pass a pre-instantiated tokenizer object.

In [ ]:
from inif.converters.text import from_texts

texts = [
    (
        "The Eiffel Tower in Paris was built in 1889."
        " It stands 330 metres tall and attracts"
        " 7 million visitors each year."
    ),
    (
        "The Eiffel Tower in Paris is one of the most"
        " visited monuments in the world."
        " About 25000 tonnes of iron were used."
    ),
    (
        "The Eiffel Tower in Paris was designed by"
        " Gustave Eiffel for the 1889 World's Fair."
        " It took 2 years to build."
    ),
]

doc = from_texts(texts, tokenizer="gpt2", min_sequence_length=5)

print(f"InifDocument: {len(doc.samples)} samples")
print(f"Common sequences found: {len(doc.sequences)}")
for seq in doc.sequences:
    print(f"  '{seq.id}': {seq.tokens}")
print(f"Model: {doc.metadata.model.name}")

## Step 2: Inspect the document structure

Each sample contains tokens (with possible sequence references), the original text, and empty slots for spans, scores, and metadata.

In [2]:
for sample in doc.samples:
    n_refs = sum(1 for t in sample.tokens if t.is_sequence_ref)
    n_flat = len(sample.tokens) - n_refs
    print(f"Sample '{sample.id}':")
    print(f"  Tokens: {len(sample.tokens)} ({n_flat} flat, {n_refs} refs)")
    print(f"  Text preview: {sample.texts[0][:60]}...")
    tok_strs = [
        f"[{t.sequence_id}]" if t.is_sequence_ref else t.token for t in sample.tokens
    ]
    print(f"  Token strings: {tok_strs}")
    print()

Sample 'sample_0':
  Tokens: 19 (18 flat, 1 refs)
  Text preview: The Eiffel Tower in Paris was built in 1889. It stands 330 m...
  Token strings: ['[seq_0]', ' was', ' built', ' in', ' 1889', '.', ' It', ' stands', ' 330', ' metres', ' tall', ' and', ' attracts', ' 7', ' million', ' visitors', ' each', ' year', '.']

Sample 'sample_1':
  Tokens: 21 (20 flat, 1 refs)
  Text preview: The Eiffel Tower in Paris is one of the most visited monumen...
  Token strings: ['[seq_0]', ' is', ' one', ' of', ' the', ' most', ' visited', ' monuments', ' in', ' the', ' world', '.', ' About', ' 25', '000', ' tonnes', ' of', ' iron', ' were', ' used', '.']

Sample 'sample_2':
  Tokens: 23 (22 flat, 1 refs)
  Text preview: The Eiffel Tower in Paris was designed by Gustave Eiffel for...
  Token strings: ['[seq_0]', ' was', ' designed', ' by', ' Gust', 'ave', ' E', 'iff', 'el', ' for', ' the', ' 1889', ' World', "'s", ' Fair', '.', ' It', ' took', ' 2', ' years', ' to', ' build', '.']



## Step 3: Tag tokens

This small notebook expands a working copy before text-level tagging because `InifDocument.tag_by_text_regex` maps concatenated text back to token positions. For large evals, keep the canonical document deduplicated: batch token-level regex strategies with `InifDocument.tag_by_regexes`, use `FlatTokenStore` for aggregate scans, and expand only the sample you need to inspect.

Two tagging modes are available:
- `InifDocument.tag_by_regex` / `InifDocument.tag_by_regexes` — match against individual token strings (fast, but misses BPE-split words)
- `InifDocument.tag_by_text_regex` — match against concatenated text and map back to constituent tokens (handles subword splits)

In [ ]:
doc_expanded = doc.expand_sequences()

# Annotate tokens containing digits (per-token regex is fine here)
doc_expanded.tag_by_regex(r"\d+", "number")

# Annotate named entities using text-level regex to handle BPE splits
# (e.g. GPT-2 splits "Eiffel" into [" E", "iff", "el"])
doc_expanded.tag_by_text_regex(r"(?i)eiffel|paris|gustave", "entity")

for sample in doc_expanded.samples:
    numbers = sample.select_by_annotation("number")
    entities = sample.select_by_annotation("entity")
    print(f"Sample '{sample.id}':")
    print(
        f"  Numbers: {[t.token for t in numbers.tokens]}"
        f" at positions {numbers.positions}"
    )
    print(
        f"  Entities: {[t.token for t in entities.tokens]}"
        f" at positions {entities.positions}"
    )
    print()

### Large-file tagging pattern

For simple token-level scans, avoid constructing or mutating `Token` objects in the hot loop. `FlatTokenStore` stores token ids/text, sample offsets, sequence provenance, and sparse annotation positions in flat arrays while the normal `.inif.json` document remains the canonical interchange format.

In [ ]:
from inif import FlatTokenStore

# Batch multiple token-level regex strategies in one pass on the canonical doc.
doc.tag_by_regexes([(r"\d+", "number")])

# For aggregate scans/statistics at larger scale, flatten into arrays.
store = FlatTokenStore.from_document(doc)
store.annotate_regexes(
    [
        (r"\d+", "number"),
        (r"(?i)eiffel|paris|gustave", "entity_hint"),
    ]
)
print(store.annotation_counts())

## Step 4: Create spans from tagged tokens

Spans are named groups of token positions stored on the sample. They are useful for marking regions of interest (e.g. an answer span, a reasoning chain).

In [ ]:
for sample in doc_expanded.samples:
    sample.create_span_from_tag("number", "numeric_tokens")
    sample.create_span_from_tag("entity", "entity_tokens")

# Verify span-based selection works
sample = doc_expanded.samples[0]
print(f"Spans on '{sample.id}':")
for span in sample.spans:
    sel = sample.select_by_span(span.name)
    print(
        f"  '{span.name}': positions={span.positions},"
        f" tokens={[t.token for t in sel.tokens]}"
    )

## Step 5: Attach mock interpretability data

In a real workflow, you would run logit lens (or another interpretability method) on the tagged positions using **nnterp** + **nnsight**:

```python
from nnterp import StandardizedTransformer
from nnterp.interventions import logit_lens

model = StandardizedTransformer("gpt2")

for sample in doc_expanded.samples:
    numbers = sample.select_by_annotation("number")
    results = logit_lens(model, sample.texts[0], token_idx=numbers.positions)
    for token, result in zip(numbers.tokens, results):
        token.set_extra("logit_lens", result)
```

Here we mock the logit lens output.

In [ ]:
import random

random.seed(42)
NUM_LAYERS = 4

for sample in doc_expanded.samples:
    for token in sample.select_by_annotation("number").tokens:
        logit_lens = {}
        for layer in range(NUM_LAYERS):
            prob = random.uniform(0.1, 0.9)
            top_tok = token.token if prob > 0.5 else "other"
            logit_lens[f"layer_{layer}"] = {
                "top_k": [{"token": top_tok, "prob": round(prob, 3)}]
            }
        token.set_extra("logit_lens", logit_lens)

# Show example
example = doc_expanded.samples[0].select_by_annotation("number").tokens[0]
print(f"Token '{example.token}' — logit lens data:")
for layer, data in example.get_extra("logit_lens", {}).items():
    top = data["top_k"][0]
    print(f"  {layer}: '{top['token']}' (prob={top['prob']})")

## Step 6: Save, validate, and reload

The enriched document (with annotations, spans, and logit lens data) can be serialized to JSON or to the indexed `.inif` archive. Extra fields on tokens and `Sample.annotations` survive the round-trip in both formats.

In [ ]:
from inif import InifDocument, validate

# Save to disk
doc_expanded.save("eiffel_tower.inif.json")

# Validate against the JSON schema
validate(doc_expanded.to_dict())
print("Schema validation passed!")

# Reload and verify
doc_reloaded = InifDocument.load("eiffel_tower.inif.json")
print(f"Reloaded: {len(doc_reloaded.samples)} samples")

# Verify extra fields survived the round-trip
reloaded_nums = doc_reloaded.samples[0].select_by_annotation("number")
rt_token = reloaded_nums.tokens[0]
assert rt_token.has_extra("logit_lens")
print(f"Logit lens data preserved for token '{rt_token.token}'")

In [ ]:
import json

# Preview the compact JSON (first sample, truncated tokens)
d = doc_expanded.to_dict(compact=True)
preview = {"metadata": d["metadata"], "samples": [d["samples"][0]]}
preview["samples"][0]["tokens"] = preview["samples"][0]["tokens"][:5]
print(json.dumps(preview, indent=2))

In [9]:
# Cleanup
import os

os.remove("eiffel_tower.inif.json")